# เทรนตัวนับเม็ดยา บน Colab (ฟรี)

เทรน YOLOv11 detection บนข้อมูล `testpills/pill-count v4` — 231 ภาพ, 21,022 กล่องยา, ถาดน้ำเงินเม็ดขาว เฉลี่ย 79 เม็ด/ภาพ

**ก่อนเริ่ม: เปิด GPU ก่อน** เมนู `Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** → Save

แล้วกด `Runtime` → `Run all` รอ ~15 นาที

---

### ⚠️ อ่านก่อน: คะแนนที่จะได้ตอนท้ายเชื่อไม่ได้

ภาพ valid ทั้ง 21 ภาพในชุดนี้ **มาจากคลิปวิดีโอเดียวกับภาพ train** (ตรวจแล้ว 21/21) เฟรมที่ 40 กับเฟรมที่ 41 ของคลิปเดียวกันแทบเป็นภาพเดียวกัน พอแบ่ง train/valid โดยสุ่มเฟรม ภาพ valid เกือบทุกใบเลยมีฝาแฝดอยู่ใน train

ผลคือ mAP จะออกมาสวยมาก แล้วไม่ได้แปลว่าอะไรเลย — เป็นปัญหาเดียวกับที่ `model2/dataset.py` ในโปรเจกต์เขียนเตือนไว้ และเป็นเหตุผลที่โมเดลเดิมรายงาน recall 1.000 แต่นับถาดจริงพลาด 2 เม็ด

**ตัวตัดสินจริงคือขั้นที่ 6** — เอาโมเดลไปนับถาดของคุณเอง ที่มันไม่เคยเห็น

## 1. เช็คว่าได้ GPU จริง

ถ้าขึ้น `NO GPU` ให้กลับไปเปิด T4 ตามด้านบนก่อน ไม่งั้นจะช้ากว่า 20 เท่า

In [ ]:
import subprocess

out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode == 0 and out.stdout.strip():
    print("GPU:", out.stdout.strip())
else:
    print("NO GPU -- Runtime > Change runtime type > T4 GPU, แล้วรันเซลล์นี้ใหม่")

## 2. ติดตั้ง ultralytics

In [ ]:
%pip install -q ultralytics

import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. อัปโหลด dataset

เลือกไฟล์ **`pill count.v4i.yolov11.zip`** (อยู่ที่ `D:\pillsort\model3\dataset\`) ขนาด 12.8 MB

เซลล์นี้จะแตกไฟล์แล้วแก้ path ใน `data.yaml` ให้เอง — ของที่ Roboflow ส่งมาเขียน `../train/images` ซึ่งชี้ออกนอกโฟลเดอร์ตัวเองและพังเสมอ

In [ ]:
import glob
import os
import zipfile

from google.colab import files

DATA_DIR = "/content/pillcount"

uploaded = files.upload()
name = next(n for n in uploaded if n.lower().endswith(".zip"))
with zipfile.ZipFile(name) as z:
    z.extractall(DATA_DIR)

# Rewrite the paths. An absolute `path` is the only form ultralytics resolves the same
# way from every working directory -- a relative one is taken against the CWD, not
# against this file.
yaml_path = os.path.join(DATA_DIR, "data.yaml")
with open(yaml_path, "w") as fh:
    fh.write(f"path: {DATA_DIR}\n"
             "train: train/images\n"
             "val: valid/images\n"
             "test: test/images\n"
             "nc: 1\n"
             "names: ['pill']\n")

for split in ("train", "valid", "test"):
    imgs = glob.glob(f"{DATA_DIR}/{split}/images/*")
    boxes = sum(len([r for r in open(t).read().splitlines() if r.strip()])
                for t in glob.glob(f"{DATA_DIR}/{split}/labels/*.txt"))
    per = boxes / len(imgs) if imgs else 0
    print(f"{split:<6} {len(imgs):>4} ภาพ   {boxes:>6} กล่อง   {per:>5.1f} เม็ด/ภาพ")

## 4. อัปโหลดภาพถาดของคุณ (ขั้นสำคัญที่สุด)

เลือก **`tray_full.jpg`** และ **`tray_sparse.jpg`** จาก `D:\pillsort\model3\samples\` (เลือกทีเดียวสองไฟล์ได้)

ภาพพวกนี้ไม่ได้อยู่ในชุดเทรน โมเดลไม่เคยเห็น — เป็นข้อสอบจริงใบเดียวที่เชื่อได้ ถ้าข้ามขั้นนี้คุณจะเหลือแค่ตัวเลข mAP ที่แปลอะไรไม่ได้

In [ ]:
BENCH_DIR = "/content/bench"
os.makedirs(BENCH_DIR, exist_ok=True)

for fname, blob in files.upload().items():
    with open(os.path.join(BENCH_DIR, fname), "wb") as fh:
        fh.write(blob)

bench = sorted(glob.glob(f"{BENCH_DIR}/*"))
print("ภาพม้านั่งที่จะใช้ทดสอบ:", [os.path.basename(p) for p in bench] or "ไม่มี -- ข้ามขั้นนี้ไปแล้ว")

## 5. เทรน

ประมาณ 10–15 นาทีบน T4

**`yolo11n` (nano) เลือกมาให้เข้ากับม้านั่ง ไม่ใช่เพราะเร็วที่สุดตอนเทรน** — เครื่องที่ร้านยาไม่มี GPU ([model1](model1/) รันบน CPU) โมเดลใหญ่กว่านี้เทรนบน Colab ก็เร็วพอกัน แต่ไปช้าตอนใช้งานจริง ถ้าอยากลองตัวใหญ่ขึ้นเปลี่ยน `yolo11n.pt` เป็น `yolo11s.pt` ได้ แล้ววัดเทียบกันที่ขั้น 6

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")     # pretrained, not the .yaml -- 231 images is far too few
                               # to train an architecture from random init.
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,                 # the size Roboflow exported at; smaller loses tablets
    batch=16,
    patience=25,               # stop early if val stops improving, to save GPU minutes
    project="/content/runs",
    name="pillcount",
    plots=True,
)

best = "/content/runs/pillcount/weights/best.pt"
print("\nweights:", best)

## 6. ข้อสอบจริง — นับถาดของคุณ

ขั้นนี้คือเหตุผลทั้งหมดของ notebook นี้

ตัวเลข mAP จากขั้น 5 เชื่อไม่ได้ (valid ปนคลิปกับ train) แต่ภาพสองใบนี้โมเดลไม่เคยเห็นจริงๆ

**ดูรูป อย่าดูแค่ตัวเลข** ถาดเต็มควรได้ราว 60 ถาดโล่งควรได้ 4 และที่สำคัญกว่าคือกรอบต้องลงบนเม็ดยา ไม่ใช่บนช่องว่าง — `pill-count/1` ตัวเดิมเคยให้เลข 63 ที่ดูเหมือนถูก ทั้งที่กรอบตกบนช่องว่างเกือบหมด

In [ ]:
import matplotlib.pyplot as plt

trained = YOLO(best)

if not bench:
    print("ไม่มีภาพม้านั่ง -- กลับไปรันขั้น 4 ก่อน")
else:
    fig, axes = plt.subplots(1, len(bench), figsize=(9 * len(bench), 7))
    axes = [axes] if len(bench) == 1 else list(axes)
    for ax, path in zip(axes, bench):
        # max_det defaults to 300, which is above the 175 of the densest training image,
        # so a full tray cannot be silently truncated here.
        r = trained.predict(path, conf=0.25, imgsz=640, max_det=1000, verbose=False)[0]
        n = len(r.boxes)
        ax.imshow(r.plot(labels=False, line_width=2)[:, :, ::-1])
        ax.set_title(f"{os.path.basename(path)}  ->  {n} เม็ด", fontsize=15)
        ax.axis("off")
        print(f"{os.path.basename(path):<20} {n:>4} เม็ด")
    plt.tight_layout()
    plt.show()

## 7. โหลด best.pt กลับเครื่อง

เอาไปวางที่ `D:\pillsort\model3\weights\` แล้วบอกผม เดี๋ยวต่อเข้ากับ `model2/count.py` ให้ — จะได้รันบนกล้องจริงแบบออฟไลน์ ไม่ต้องพึ่งเน็ตและไม่เสียเครดิต

In [ ]:
files.download(best)